# Fraud Detection System — Exploratory Data Analysis
This notebook analyzes transaction, user, and claim data to understand patterns, distributions, and fraud signals.

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import sys
from dotenv import load_dotenv
from sqlalchemy import create_engine

sys.path.append(os.path.abspath('..'))
load_dotenv('../.env')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('Setup complete!')

## 2. Load Data

In [ ]:
DATABASE_URL = os.getenv('DATABASE_URL')

if DATABASE_URL:
    engine = create_engine(DATABASE_URL)
    transactions_df = pd.read_sql('SELECT * FROM transactions', engine)
    users_df        = pd.read_sql('SELECT * FROM users', engine)
    claims_df       = pd.read_sql('SELECT * FROM claims', engine)
    print('Loaded data from PostgreSQL database')
else:
    print('DATABASE_URL not found — using synthetic training data as fallback')
    transactions_df = pd.DataFrame({
        'amount':            [100, 200, 50, 300, 150, 75, 250, 400, 180, 90,
                              5000, 7000, 10000, 8000, 6000, 9000, 5500, 7500, 6500, 8500],
        'amount_deviation':  [1.0, 1.1, 0.9, 1.2, 1.0, 0.8, 1.3, 1.5, 1.1, 0.9,
                              10.0, 15.0, 20.0, 12.0, 8.0, 18.0, 11.0, 14.0, 9.0, 16.0],
        'is_new_location':   [0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
                              1, 1, 1, 1, 0, 1, 1, 0, 1, 1],
        'is_flagged_device': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
                              0, 1, 1, 0, 1, 1, 0, 1, 0, 1],
        'velocity':          [1, 2, 1, 3, 2, 1, 2, 1, 3, 2,
                              1, 2, 8, 6, 10, 7, 3, 5, 4, 9],
        'is_fraud':          [0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
                              1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        'risk_score':        [0.1, 0.15, 0.1, 0.2, 0.15, 0.1, 0.2, 0.25, 0.15, 0.1,
                              0.8, 0.9, 1.0, 0.85, 0.75, 0.95, 0.8, 0.88, 0.78, 0.92],
        'risk_level':        ['LOW']*10 + ['HIGH']*10,
        'decision':          ['ALLOW']*10 + ['REJECT']*10,
    })
    claims_df = pd.DataFrame()

print(f'Transactions loaded: {len(transactions_df)} rows')

## 3. Section 1 — Data Types & Missing Values

In [ ]:
# Check dtypes and missing values for ALL columns at once
print('=== DataFrame Info ===')
transactions_df.info()

In [ ]:
print('=== Missing Values Per Column ===')
print(transactions_df.isnull().sum())

In [ ]:
# Automated missing value handler — only fixes columns that actually have missing values
print('Fixing missing values automatically...\n')

for col in transactions_df.columns:
    missing_count = transactions_df[col].isnull().sum()
    if missing_count > 0:
        if transactions_df[col].dtype == 'float64':
            transactions_df[col].fillna(transactions_df[col].mean(), inplace=True)
            print(f'  {col}: filled {missing_count} missing values with mean')
        elif transactions_df[col].dtype == 'object':
            transactions_df[col].fillna(transactions_df[col].mode()[0], inplace=True)
            print(f'  {col}: filled {missing_count} missing values with mode')
        elif transactions_df[col].dtype == 'bool':
            transactions_df[col].fillna(False, inplace=True)
            print(f'  {col}: filled {missing_count} missing values with False')
        elif transactions_df[col].dtype == 'int64':
            transactions_df[col].fillna(0, inplace=True)
            print(f'  {col}: filled {missing_count} missing values with 0')
        else:
            transactions_df[col].fillna(method='ffill', inplace=True)
            print(f'  {col}: filled {missing_count} missing values with forward fill')

print('\nDone!')

In [ ]:
# Fix datetime column dtype if loaded as string
if 'created_at' in transactions_df.columns:
    if transactions_df['created_at'].dtype == 'object':
        transactions_df['created_at'] = pd.to_datetime(transactions_df['created_at'])
        print('created_at converted to datetime')

print('=== Final dtypes after cleaning ===')
print(transactions_df.dtypes)

## 4. Section 2 — Transaction Overview

In [ ]:
print('=== Basic Statistics ===')
transactions_df[['amount', 'risk_score']].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(transactions_df['amount'], bins=30, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount')

sns.boxplot(x=transactions_df['amount'], ax=axes[1], color='lightcoral')
axes[1].set_title('Transaction Amount Boxplot')

plt.tight_layout()
plt.show()

## 5. Section 3 — Fraud vs Legitimate (Class Imbalance)

In [ ]:
fraud_counts = transactions_df['is_fraud'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(fraud_counts, labels=['Legitimate', 'Fraud'],
            autopct='%1.1f%%', colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('Fraud vs Legitimate Transactions')

sns.countplot(x='is_fraud', data=transactions_df,
              palette=['#4CAF50', '#F44336'], ax=axes[1])
axes[1].set_title('Transaction Count by Fraud Label')
axes[1].set_xticklabels(['Legitimate', 'Fraud'])

plt.tight_layout()
plt.show()

total = len(transactions_df)
print(f'Fraud rate: {fraud_counts.get(1, 0) / total * 100:.1f}%')
print(f'Legitimate: {fraud_counts.get(0, 0)} | Fraud: {fraud_counts.get(1, 0)}')

## 6. Section 4 — Fraud Signal Analysis

In [ ]:
# Amount and risk score by fraud label
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='is_fraud', y='amount', data=transactions_df,
            palette=['#4CAF50', '#F44336'], ax=axes[0])
axes[0].set_title('Transaction Amount by Fraud Label')
axes[0].set_xticklabels(['Legitimate', 'Fraud'])

sns.boxplot(x='is_fraud', y='risk_score', data=transactions_df,
            palette=['#4CAF50', '#F44336'], ax=axes[1])
axes[1].set_title('Risk Score by Fraud Label')
axes[1].set_xticklabels(['Legitimate', 'Fraud'])

plt.tight_layout()
plt.show()

In [ ]:
# Fraud rate by binary signals (is_new_location, is_flagged_device)
signal_cols = [c for c in ['is_new_location', 'is_flagged_device']
               if c in transactions_df.columns]

if signal_cols:
    fig, axes = plt.subplots(1, len(signal_cols), figsize=(14, 5))
    if len(signal_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, signal_cols):
        fraud_rate = transactions_df.groupby(col)['is_fraud'].mean() * 100
        fraud_rate.plot(kind='bar', ax=ax, color=['#4CAF50', '#F44336'], rot=0)
        ax.set_title(f'Fraud Rate by {col}')
        ax.set_ylabel('Fraud Rate (%)')
        ax.set_xticklabels(['No', 'Yes'])

    plt.tight_layout()
    plt.show()

In [ ]:
# Transaction velocity vs fraud
if 'velocity' in transactions_df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x='is_fraud', y='velocity', data=transactions_df,
                palette=['#4CAF50', '#F44336'])
    plt.title('Transaction Velocity by Fraud Label')
    plt.xticks([0, 1], ['Legitimate', 'Fraud'])
    plt.show()

## 7. Section 5 — Risk Score & Risk Level Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(transactions_df['risk_score'], bins=20, kde=True,
             ax=axes[0], color='purple')
axes[0].axvline(0.3, color='orange', linestyle='--', label='Low/Medium (0.3)')
axes[0].axvline(0.7, color='red',    linestyle='--', label='Medium/High (0.7)')
axes[0].set_title('Risk Score Distribution')
axes[0].legend()

if 'risk_level' in transactions_df.columns:
    risk_counts = transactions_df['risk_level'].value_counts()
    colors = {'LOW': '#4CAF50', 'MEDIUM': '#FF9800', 'HIGH': '#F44336'}
    axes[1].pie(
        risk_counts,
        labels=risk_counts.index,
        autopct='%1.1f%%',
        colors=[colors.get(k, 'grey') for k in risk_counts.index]
    )
    axes[1].set_title('Risk Level Distribution')

plt.tight_layout()
plt.show()

## 8. Section 6 — Decision Distribution

In [ ]:
if 'decision' in transactions_df.columns:
    decision_order  = ['ALLOW', 'MANUAL_CHECK', 'REJECT']
    decision_colors = {'ALLOW': '#4CAF50', 'MANUAL_CHECK': '#FF9800', 'REJECT': '#F44336'}
    order   = [d for d in decision_order if d in transactions_df['decision'].unique()]
    palette = [decision_colors[d] for d in order]

    plt.figure(figsize=(8, 5))
    sns.countplot(x='decision', data=transactions_df, order=order, palette=palette)
    plt.title('Transaction Decision Distribution')
    plt.xlabel('Decision')
    plt.ylabel('Count')
    plt.show()

    print(transactions_df['decision'].value_counts())

## 9. Section 7 — Feature Correlation Heatmap

In [ ]:
feature_cols = [c for c in ['amount', 'amount_deviation', 'is_new_location',
                              'is_flagged_device', 'velocity', 'risk_score', 'is_fraud']
                if c in transactions_df.columns]

plt.figure(figsize=(10, 7))
corr = transactions_df[feature_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 10. Section 8 — Claims Analysis

In [ ]:
if len(claims_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    status_colors = {
        'APPROVED': '#4CAF50', 'REJECTED': '#F44336',
        'PENDING': '#2196F3',  'MANUAL_REVIEW': '#FF9800'
    }
    status_counts = claims_df['status'].value_counts()
    axes[0].pie(
        status_counts,
        labels=status_counts.index,
        autopct='%1.1f%%',
        colors=[status_colors.get(k, 'grey') for k in status_counts.index]
    )
    axes[0].set_title('Claim Status Distribution')

    sns.histplot(claims_df['amount'], bins=20, kde=True, ax=axes[1], color='steelblue')
    axes[1].set_title('Claimed Amount Distribution')

    plt.tight_layout()
    plt.show()
else:
    print('No claims data available — connect to DB or file some claims first')

## 11. Section 9 — ML Model Feature Importance

In [ ]:
MODEL_PATH = '../app/ML/model.pkl'

if os.path.exists(MODEL_PATH):
    model = joblib.load(MODEL_PATH)
    feature_names = ['amount', 'amount_deviation', 'is_new_location',
                     'is_flagged_device', 'velocity']

    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=True)

    plt.figure(figsize=(8, 5))
    plt.barh(importance_df['feature'], importance_df['importance'], color='steelblue')
    plt.title('ML Model — Feature Importance (RandomForest)')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.show()

    print(importance_df.sort_values('importance', ascending=False).to_string(index=False))
else:
    print(f'Model not found — run python app/ML/train_model.py first')

## 12. Summary
- **Section 1:** Dtypes verified, missing values handled automatically per column dtype
- **Section 2:** Transaction amount distribution and basic statistics
- **Section 3:** Fraud vs legitimate class imbalance check
- **Section 4:** Fraud signals — amount, new location, flagged device, velocity
- **Section 5:** Risk score distribution with LOW / MEDIUM / HIGH boundaries
- **Section 6:** Decision breakdown — ALLOW / MANUAL_CHECK / REJECT
- **Section 7:** Feature correlation heatmap
- **Section 8:** Claims analysis — status distribution and claimed amounts
- **Section 9:** ML model feature importance from trained RandomForest